## Structured Output

Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

In [2]:
import os 
from dotenv import load_dotenv
load_dotenv()

os.environ["ANTHROPIC_API_KEY"] = os.getenv("ANTHROPIC_API_KEY")

In [1]:
from langchain.messages import SystemMessage, HumanMessage, AIMessage

from langchain.chat_models import init_chat_model

model_name = "claude-haiku-4-5-20251001"


model = init_chat_model("claude-haiku-4-5-20251001")


## PYDANTIC

Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [7]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title : str = Field(description="The Title of Movie")
    year : int = Field(description="The year Movie is released")
    director : str = Field(description = "The Director of Movie")
    rating : int = Field(description="Movie Rating out of 10")

model_with_structure = model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatAnthropic(metadata={'lc_versions': {'langchain-core': '1.6.3', 'langchain': '1.4.0', 'langchain-anthropic': '1.7.2'}}, profile={'name': 'Claude Haiku 4.5', 'release_date': '2025-10-15', 'last_updated': '2025-10-15', 'open_weights': False, 'max_input_tokens': 200000, 'max_output_tokens': 64000, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': False, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_call_streaming': True}, model='claude-haiku-4-5-20251001', max_tokens=64000, anthropic_api_url='https://api.anthropic.com', anthropic_api_key=SecretStr('**********'), model_kwargs={}), kwargs={'tools': [{'name': 'Movie', 'input_schema': {'properties': {'title': {'descrip

In [8]:
model.invoke("Provide details about the model Inception")

AIMessage(content='# Inception Model - Comprehensive Overview\n\n## What is Inception?\n\nInception is a family of deep convolutional neural networks developed by Google, designed to be computationally efficient while achieving high accuracy on image classification tasks. The first version was introduced in 2014 and has since evolved through multiple iterations.\n\n## Key Versions\n\n### **Inception-v1 (GoogLeNet) - 2014**\n- Winner of ImageNet Large Scale Visual Recognition Challenge (ILSVRC) 2014\n- Introduced the concept of **multi-scale feature extraction**\n- 22 layers deep (much deeper than contemporaries)\n- Parameters: ~6.5 million\n\n### **Inception-v2 & v3 - 2015**\n- Improved training techniques and architectural refinements\n- Better hyperparameter choices\n- More efficient and accurate\n- Introduced **Batch Normalization** (v2)\n\n### **Inception-v4 & Inception-ResNet - 2016**\n- Combined Inception modules with **residual connections**\n- Enhanced performance on ImageNet\n

In [9]:
model_with_structure.invoke("Provide details about the model Inception")

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8)

In [10]:
response = model_with_structure.invoke("Provide details about the model Inception")
print(response)

title='Inception' year=2010 director='Christopher Nolan' rating=8


In [11]:
## Nested Structure

In [13]:
from pydantic import BaseModel, Field


class Actor(BaseModel):
    name: str
    role : str 

class MovieDetails(BaseModel):
    title : str
    year : int
    cast : list[Actor]
    genres : list[str]
    budget : float | None = Field(description="Budget in Millions USD")


In [14]:
model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the model Inception")
print(response)

title='Inception' year=2010 cast=[Actor(name='Leonardo DiCaprio', role='Cobb'), Actor(name='Marion Cotillard', role='Mal'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Tom Hardy', role='Eames')] genres=['Science Fiction', 'Action', 'Thriller'] budget=160.0


## TypedDict
TypedDict provides a simpler alternative using Python’s built-in typing, ideal when you don’t need runtime validation.

In [ ]:
from typing_extensions import TypedDict,Annotated

## ... represent optional parameter kept as like that

class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]


model_withtypedict=model.with_structured_output(MovieDict)
response=model_withtypedict.invoke("Please provide the details of the movie avengers")
response

{'title': 'The Avengers',
 'year': 2012,
 'director': 'Joss Whedon',
 'rating': 8.0}

In [16]:
## Can invoke pytdentic class without run time validation using typedict

class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Avengers")
response

{'title': 'Avengers',
 'year': 2012,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Iron Man/Tony Stark'},
  {'name': 'Chris Evans', 'role': 'Captain America/Steve Rogers'},
  {'name': 'Chris Hemsworth', 'role': 'Thor'},
  {'name': 'Mark Ruffalo', 'role': 'Hulk/Bruce Banner'},
  {'name': 'Scarlett Johansson', 'role': 'Black Widow/Natasha Romanoff'},
  {'name': 'Jeremy Renner', 'role': 'Hawkeye/Clint Barton'}],
 'genres': ['Action', 'Adventure', 'Sci-Fi'],
 'budget': 220000000}

In [17]:
model.profile

{'name': 'Claude Haiku 4.5',
 'release_date': '2025-10-15',
 'last_updated': '2025-10-15',
 'open_weights': False,
 'max_input_tokens': 200000,
 'max_output_tokens': 64000,
 'text_inputs': True,
 'image_inputs': True,
 'audio_inputs': False,
 'pdf_inputs': True,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': False,
 'attachment': True,
 'temperature': True,
 'image_url_inputs': True,
 'pdf_tool_message': True,
 'image_tool_message': True,
 'tool_call_streaming': True}

## DataClasses
A data class is a class typically containing mainly data, although there aren’t really any restrictions. You create it using the @dataclass decorator

In [18]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent


class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model=model_name,
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='a361638f-c8a0-40ea-b4f1-de7217bef1bb'),
  AIMessage(content='{"name":"John Doe","email":"john@example.com","phone":"(555) 123-4567"}', additional_kwargs={}, response_metadata={'id': 'msg_011Cf9TQGz1UpnTBMWqt7kxD', 'container': None, 'model': 'claude-haiku-4-5-20251001', 'stop_details': None, 'stop_reason': 'end_turn', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'not_available', 'input_tokens': 298, 'output_tokens': 29, 'output_tokens_details': None, 'server_tool_use': None, 'service_tier': 'standard'}, 'model_name': 'claude-haiku-4-5-20251001', 'model_provider': 'anthropic'}, id='lc_run--01a0b08c-59ce-7412-a49f-1b006d2ea8da-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={

In [19]:
result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

In [21]:
## Typedict
from typing_extensions import TypedDict
from langchain.agents import create_agent


class ContactInfo(TypedDict):
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person

agent = create_agent(
    model=model_name,
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]
# {'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

In [22]:
## Dataclass

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person


agent = create_agent(
    model=model_name,
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')